# Multi-objective Diet Optimization — main.ipynb

This is the **main notebook** for the project. It loads the **real MySQL dataset** (no mock/sample data), runs NSGA-II and PSO (via `inspyred`), and produces plots + replicate statistics.

## Manual setup required

1) Import the provided dump into MySQL: `data/food_database_dump.sql`
2) Create a `.env` file (copy `.env.example`) and fill in your credentials
3) Start MySQL and make sure it’s reachable (host/port/user/password)

In [ ]:
# Make `src/` importable so we can `import dietopt` and `import utilidades`
from __future__ import annotations

import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "src").is_dir():
            return p
    raise RuntimeError("Could not find repo root containing 'src/'")


repo_root = _find_repo_root(Path.cwd())
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print("cwd:", Path.cwd())
print("repo_root:", repo_root)
print("src_root:", src_root)
print("python:", sys.executable)

In [ ]:
import matplotlib.pyplot as plt

from dietopt.data import load_foods_from_db, load_subjects_from_db, require_db_env
from dietopt.plotting import plot_convergence, plot_pareto
from dietopt.algorithms.nsga2 import run_nsga2
from dietopt.algorithms.pso import run_pso_scalarized

require_db_env()

In [ ]:
foods = load_foods_from_db()
subjects = load_subjects_from_db()
subject = subjects[0]

print("n_foods =", len(foods))
print("subject =", subject)

In [ ]:
# Quick demo run (keep generations small for interactivity)
nsga = run_nsga2(foods, subject.edad, subject.calorias, pop_size=60, max_generations=40, seed=1)
pso  = run_pso_scalarized(foods, subject.edad, subject.calorias, pop_size=50, max_generations=40, seed=1)

print('NSGA-II best_f (representative):', nsga['best_f'])
print('PSO best_f:', pso['best_f'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_pareto(nsga['front'], axes[0], label='NSGA-II final pop')
plot_pareto(pso['front'], axes[0], label='PSO best')
axes[0].legend()

plot_convergence(nsga['trace'], axes[1], label='NSGA-II')
plot_convergence(pso['trace'], axes[1], label='PSO')
axes[1].legend()

plt.tight_layout()
plt.show()

## How to run the required 30+ executions

Use `dietopt.experiments.run_replicates(...)` and set `n_runs=30` (or more).
Then compare algorithms/configurations using the Mann–Whitney U test (see `mann_whitney_u_p_value`).

In [ ]:
import pandas as pd
from IPython.display import display

from dietopt.experiments import mann_whitney_u_p_value, run_replicates

# For the report: set n_runs = 30 (or more).
n_runs = 5

nsga_runs = run_replicates(
    "NSGA-II",
    run_nsga2,
    n_runs=n_runs,
    seed0=100,
    comida_bd=foods,
    edad=subject.edad,
    ctarget=subject.calorias,
    pop_size=60,
    max_generations=40,
)

pso_runs = run_replicates(
    "PSO",
    run_pso_scalarized,
    n_runs=n_runs,
    seed0=200,
    comida_bd=foods,
    edad=subject.edad,
    ctarget=subject.calorias,
    pop_size=50,
    max_generations=40,
)

df = pd.DataFrame([r.__dict__ for r in (nsga_runs + pso_runs)])
display(df.groupby("algorithm")[["f1", "f2", "runtime_s"]].agg(["mean", "std", "min", "max"]))

p_f1 = mann_whitney_u_p_value(df[df.algorithm == "NSGA-II"].f1, df[df.algorithm == "PSO"].f1)
p_f2 = mann_whitney_u_p_value(df[df.algorithm == "NSGA-II"].f2, df[df.algorithm == "PSO"].f2)
print("Mann–Whitney p-value (f1):", p_f1)
print("Mann–Whitney p-value (f2):", p_f2)